In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
from pathlib import Path

In [2]:
url_listings = "https://data.insideairbnb.com/colombia/dc/bogot%C3%A1/2026-06-21/visualisations/listings.csv"
url_listings_det = "https://data.insideairbnb.com/colombia/dc/bogot%C3%A1/2026-06-21/data/listings.csv.gz"

listings = pd.read_csv(url_listings).rename(columns={"neighbourhood": "neighborhood"})
listings_det = pd.read_csv(url_listings_det, compression="gzip")

In [3]:
listings.columns

Index(['id', 'name', 'host_id', 'host_profile_id', 'host_name',
       'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude',
       'room_type', 'price', 'minimum_nights', 'number_of_reviews',
       'last_review', 'reviews_per_month', 'calculated_host_listings_count',
       'availability_365', 'number_of_reviews_ltm', 'license'],
      dtype='str')

In [3]:
a_eliminar = ['host_id', 'host_profile_id','host_name','number_of_reviews',
       'last_review', 'reviews_per_month', 'calculated_host_listings_count',
       'number_of_reviews_ltm', 'license','neighbourhood_group']

# Eliminar columnas innecesarias
listings = listings.drop(a_eliminar, axis=1)

In [5]:
listings.head()

,id,name,neighbourhood,latitude,longitude,room_type,price,minimum_nights,availability_365
0,27841,Rosales Luxury Large 2BR Boutique Apt,Chapinero,4.657580,-74.053260,Entire home/apt,284294.0,28.0,82
1,65762,Apartaestudios La Candelaria,Candelaria,4.594850,-74.071890,Entire home/apt,215000.0,1.0,236
2,124029,Exclusive Loft in 93 park w Gym,Chapinero,4.679779,-74.055817,Entire home/apt,273877.0,10.0,318
3,142658,"Amazing view and location, Bogotá!",Santa Fe,4.618660,-74.069370,Entire home/apt,184317.0,30.0,87
4,152126,The Place to BE,Suba,4.707640,-74.056730,Private room,128356.0,30.0,365


In [6]:
listings.info()

<class 'pandas.DataFrame'>
RangeIndex: 19187 entries, 0 to 19186
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                19187 non-null  int64  
 1   name              19187 non-null  str    
 2   neighbourhood     19187 non-null  str    
 3   latitude          19187 non-null  float64
 4   longitude         19187 non-null  float64
 5   room_type         19187 non-null  str    
 6   price             18992 non-null  float64
 7   minimum_nights    19183 non-null  float64
 8   availability_365  19187 non-null  int64  
dtypes: float64(4), int64(2), str(3)
memory usage: 1.3 MB


In [7]:
# analisis de nulos 
listings.isnull().sum()

id                    0
name                  0
neighbourhood         0
latitude              0
longitude             0
room_type             0
price               195
minimum_nights        4
availability_365      0
dtype: int64

In [4]:
# Eliminar nulos de precio
listings = listings.dropna(subset=["price", "minimum_nights"])

In [9]:
listings.duplicated().sum()

np.int64(0)

In [10]:
columnas_texto = listings.select_dtypes(include=["object", "string"]).columns

for columna in columnas_texto:
    print(f"\nValue counts de {columna}:")
    print(listings[columna].value_counts(dropna=False))


Value counts de name:
name
Impecable habitación con baño privado                 22
Nueva habitación confort con baño privado             18
Atlantis Home Suites Zona T                           17
Apartamento nuevo y amoblado con ubicación ideal      17
Habitación con baño privado                           14
                                                      ..
Moderno apartaestuido en zona central                  1
Apartaestudio cómodo y equipado, reserva inmediata     1
Habitación en Bogotá, Excelente ubicación              1
Apartaestudio tipo Hotel Diseño Moderno y Elegante     1
Hotel dream luxury                                     1
Name: count, Length: 17954, dtype: int64

Value counts de neighbourhood:
neighbourhood
Chapinero             5055
Usaquen               3541
Teusaquillo           2762
Santa Fe              1437
Suba                  1255
Engativa              1199
Barrios Unidos        1027
Fontibon               956
Candelaria             697
Kennedy        

In [5]:
#Eliminar los reistros donde room_type sea igual Hotel room
listings = listings[listings["room_type"] != "Hotel room"]

In [12]:
listings.iloc[:,6:9].describe().T

,count,mean,std,min,25%,50%,75%,max
price,18955.0,446417.393827,8.837661e+06,19181.0,109944.0,161432.0,230281.0,448176567.0
minimum_nights,18955.0,6.912688,1.094013e+01,1.0,1.0,1.0,3.0,32.0
availability_365,18955.0,307.001846,8.186163e+01,4.0,269.0,349.0,363.0,365.0


In [6]:
#Quedarnos con filas donde el precio sea mayor de 50000 y menor de 1000000
listings = listings[(listings["price"] >= 50000) & (listings["price"] <= 1000000)]

In [14]:
listings_det.info()

<class 'pandas.DataFrame'>
RangeIndex: 19187 entries, 0 to 19186
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            19187 non-null  int64  
 1   listing_url                                   19187 non-null  str    
 2   scrape_id                                     19187 non-null  int64  
 3   last_scraped                                  19187 non-null  str    
 4   source                                        19187 non-null  str    
 5   name                                          19187 non-null  str    
 6   description                                   18664 non-null  str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   19187 non-null  str    
 9   host_id                                       19187 non-null  int64  
 1

In [7]:
a_inclur = ['id',
            'description', 
            'accommodates',
            'bathrooms',
            'bedrooms',
            'beds',
            'estimated_occupancy_l365d',
            'review_scores_location']

listings_det = listings_det[a_inclur]

In [16]:
listings_det.info()

<class 'pandas.DataFrame'>
RangeIndex: 19187 entries, 0 to 19186
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         19187 non-null  int64  
 1   description                18664 non-null  str    
 2   accommodates               19187 non-null  int64  
 3   bathrooms                  18121 non-null  float64
 4   bedrooms                   16228 non-null  float64
 5   beds                       18896 non-null  float64
 6   estimated_occupancy_l365d  19187 non-null  int64  
 7   review_scores_location     14921 non-null  float64
dtypes: float64(4), int64(3), str(1)
memory usage: 1.2 MB


In [17]:
#Analisis de nulos
listings_det.isnull().sum()

id                              0
description                   523
accommodates                    0
bathrooms                    1066
bedrooms                     2959
beds                          291
estimated_occupancy_l365d       0
review_scores_location       4266
dtype: int64

In [8]:
# Eliminar registros con estimated_occupancy_l365d = 0
listings_det = listings_det[listings_det["estimated_occupancy_l365d"] != 0]

In [9]:
#Eliminar registros con accommodates mayor a 8
listings_det = listings_det[listings_det["accommodates"] <= 8]

In [10]:
#Eliminar registros con mas de 5 baños
listings_det = listings_det[listings_det["bathrooms"] <= 5]

In [11]:
#Eliminar registros con mas de 5 habitaciones
listings_det = listings_det[listings_det["bedrooms"] <= 5]

In [12]:
#Eliminar registros con mas de 6 camas
listings_det = listings_det[listings_det["beds"] <= 6]

In [23]:
#comprobar no existen duplicados
listings_det.duplicated().sum()

np.int64(0)

In [13]:
import re

m2_bogota_path = Path("../datos/brutos/m2_bogota.xlsx")
m2_bogota = pd.read_excel(m2_bogota_path)



In [14]:
m2_bogota = m2_bogota.rename(columns={
    m2_bogota.columns[0]: "neighborhood",
    m2_bogota.columns[1]: "precio"
})

m2_bogota["neighborhood"] = m2_bogota["neighborhood"].astype(str).str.strip()

def limpiar_precio(valor):
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, (int, float, np.number)):
        return float(valor)

    texto = str(valor).strip()
    texto = re.sub(r"[^\d,.-]", "", texto)

    if "," in texto and "." in texto:
        if texto.rfind(",") > texto.rfind("."):
            texto = texto.replace(".", "").replace(",", ".")
        else:
            texto = texto.replace(",", "")
    elif "," in texto:
        if texto.count(",") == 1 and len(texto.split(",")[-1]) <= 2:
            texto = texto.replace(",", ".")
        else:
            texto = texto.replace(",", "")
    elif texto.count(".") > 1 or ("." in texto and len(texto.split(".")[-1]) > 2):
        texto = texto.replace(".", "")

    return pd.to_numeric(texto, errors="coerce")

m2_bogota["precio"] = m2_bogota["precio"].apply(limpiar_precio).round().astype("Int64")
m2_bogota.loc[m2_bogota["neighborhood"] == "Sumapaz", "precio"] = 0

In [15]:
import sqlite3
from pandas.api.types import is_bool_dtype, is_float_dtype, is_integer_dtype

sqlite_path = Path("../datos/intermedios/mercado_inmobiliario_bogota.sqlite")
sqlite_path.parent.mkdir(parents=True, exist_ok=True)
try:
    conn.close()
except NameError:
    pass
except Exception:
    pass

listings_sql = listings.copy()
listings_det_sql = listings_det[listings_det["id"].isin(listings_sql["id"])].copy()
m2_bogota_sql = m2_bogota.copy()

assert listings_sql["id"].is_unique, "La clave id de listings debe ser unica"
assert listings_det_sql["id"].is_unique, "La clave id de listings_det debe ser unica"
assert m2_bogota_sql["neighborhood"].is_unique, "La clave neighborhood de m2_bogota debe ser unica"
assert listings_det_sql["id"].isin(listings_sql["id"]).all(), "Hay ids en listings_det que no existen en listings"
assert listings_sql["neighborhood"].isin(m2_bogota_sql["neighborhood"]).all(), "Hay barrios en listings que no existen en m2_bogota"

def preparar_para_sqlite(df):
    return df.astype(object).where(pd.notna(df), None)

def tipo_sqlite(serie):
    if is_bool_dtype(serie) or is_integer_dtype(serie):
        return "INTEGER"
    if is_float_dtype(serie):
        return "REAL"
    return "TEXT"

def crear_tabla(conn, nombre_tabla, df, primary_key=None, foreign_keys=None):
    foreign_keys = foreign_keys or []
    columnas = []

    for columna in df.columns:
        definicion = '"{}" {}'.format(columna, tipo_sqlite(df[columna]))
        if primary_key == columna:
            definicion += " PRIMARY KEY"
        columnas.append(definicion)

    for columna, tabla_ref, columna_ref in foreign_keys:
        columnas.append('FOREIGN KEY("{}") REFERENCES "{}"("{}")'.format(columna, tabla_ref, columna_ref))

    sentencia = 'CREATE TABLE "{}" ({})'.format(nombre_tabla, ', '.join(columnas))
    conn.execute(sentencia)

def insertar_datos(conn, nombre_tabla, df):
    columnas_sql = ', '.join('"{}"'.format(columna) for columna in df.columns)
    placeholders = ', '.join(['?'] * len(df.columns))
    registros = preparar_para_sqlite(df).itertuples(index=False, name=None)
    conn.executemany(
        'INSERT INTO "{}" ({}) VALUES ({})'.format(nombre_tabla, columnas_sql, placeholders),
        registros
    )

with sqlite3.connect(sqlite_path) as conn:
    conn.execute("PRAGMA foreign_keys = OFF")
    conn.execute("DROP VIEW IF EXISTS vw_listings_completo")
    conn.execute("DROP TABLE IF EXISTS listings_det")
    conn.execute("DROP TABLE IF EXISTS listings")
    conn.execute("DROP TABLE IF EXISTS m2_bogota")
    conn.execute("PRAGMA foreign_keys = ON")

    crear_tabla(conn, "m2_bogota", m2_bogota_sql, primary_key="neighborhood")
    crear_tabla(conn, "listings", listings_sql, primary_key="id", foreign_keys=[("neighborhood", "m2_bogota", "neighborhood")])
    crear_tabla(conn, "listings_det", listings_det_sql, primary_key="id", foreign_keys=[("id", "listings", "id")])

    insertar_datos(conn, "m2_bogota", m2_bogota_sql)
    insertar_datos(conn, "listings", listings_sql)
    insertar_datos(conn, "listings_det", listings_det_sql)

    conn.execute(
        "CREATE VIEW vw_listings_completo AS "
        "SELECT l.*, ld.description, ld.accommodates, ld.bathrooms, ld.bedrooms, "
        "ld.beds, ld.estimated_occupancy_l365d, ld.review_scores_location, "
        "m.precio AS precio_m2 FROM listings l "
        "LEFT JOIN listings_det ld ON l.id = ld.id "
        "LEFT JOIN m2_bogota m ON l.neighborhood = m.neighborhood"
    )

    resumen_tablas = pd.read_sql_query(
        """
        SELECT 'm2_bogota' AS tabla, COUNT(*) AS filas FROM m2_bogota
        UNION ALL
        SELECT 'listings', COUNT(*) FROM listings
        UNION ALL
        SELECT 'listings_det', COUNT(*) FROM listings_det
        """,
        conn
    )

print(f"Base SQLite creada en: {sqlite_path.resolve()}")
resumen_tablas

Base SQLite creada en: D:\Users\LAURA PEREZ\Desktop\CIENCIA DE DATOS\ANALISIS MERCADO INMOBILIARIO ALQUILER TURISMO\datos\intermedios\mercado_inmobiliario_bogota.sqlite


,tabla,filas
0,m2_bogota,20
1,listings,17969
2,listings_det,9687
